# Laboratório 07 — Especialização de LLMs com LoRA e QLoRA

> Pipeline completo de especialização de LLM usando **Groq API (Llama 3)** em todos os passos.  
> Domínio: **Programação / Tecnologia da Informação**

1 - Geração do Dataset Sintético; Groq API → `.jsonl`

2 - Simulação da Quantização; BitsAndBytesConfig (referência)

3 - Configuração do LoRA; LoraConfig (referência) + system prompt especializado

4 - Pipeline de Treinamento; Few-shot SFT via Groq + avaliação no dataset de teste

> **Nota de IA:** Partes geradas/complementadas com IA, revisadas por Ingrid.

## Instalação de Dependências

In [2]:
%pip install -q groq
%pip install -q torch transformers bitsandbytes peft trl datasets
%pip install -q python-dotenv
print("Dependências instaladas!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Dependências instaladas!


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Configuração Global

In [4]:
import os, json, random, time
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.environ.get("api_key", "")


MODEL_ID       = "llama-3.1-8b-instant"   # modelo Groq
OUTPUT_TRAIN   = "dataset_train.jsonl"
OUTPUT_TEST    = "dataset_test.jsonl"
ADAPTER_DIR    = "./groq-lora-adapter"
TRAIN_RATIO    = 0.90
TOTAL_PAIRS    = 55

client = Groq(api_key=GROQ_API_KEY)
print(f" Cliente Groq inicializado | Modelo: {MODEL_ID}")

 Cliente Groq inicializado | Modelo: llama-3.1-8b-instant


## Passo 1 — Engenharia de Dados Sintéticos

Gera **55 pares** instrução/resposta sobre Programação/TI via **Groq API**,  
divide em **90% treino / 10% teste** e salva em `.jsonl`.


In [9]:
TOPICS = [
    "Python básico (listas, dicionários, funções)",
    "Python orientado a objetos (classes, herança, polimorfismo)",
    "Estruturas de dados (pilha, fila, árvore, grafo)",
    "Algoritmos de ordenação (bubble sort, merge sort, quicksort)",
    "Complexidade de algoritmos (Big-O)",
    "SQL e bancos de dados relacionais",
    "Git e controle de versão",
    "APIs REST e HTTP",
    "Docker e containers",
    "Redes de computadores (TCP/IP, DNS, HTTP/HTTPS)",
    "Segurança da informação (OWASP, criptografia básica)",
    "Design patterns (Singleton, Factory, Observer)",
]

SYSTEM_PROMPT_DATASET = """Você é um professor especialista em programação e TI.
Gere EXATAMENTE um par de instrução e resposta sobre o tema fornecido.
Responda APENAS em formato JSON válido, sem texto extra, sem markdown, assim:
{"instruction": "pergunta ou tarefa clara sobre o tema", "response": "resposta técnica completa e didática (mínimo 3 linhas)"}"""

def generate_pair(topic: str):
    try:
        chat = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_DATASET},
                {"role": "user",   "content": f"Tema: {topic}"},
            ],
            temperature=0.85,
            max_tokens=512,
        )
        raw = chat.choices[0].message.content.strip()
        # Remove blocos markdown se presentes
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        pair = json.loads(raw.strip())
        assert "instruction" in pair and "response" in pair
        return pair
    except Exception as e:
        print(f"  [AVISO] Erro: {e}")
        return None

print("Funções de geração definidas.")

Funções de geração definidas.


In [10]:
pairs = []
attempts, max_attempts = 0, TOTAL_PAIRS * 3

while len(pairs) < TOTAL_PAIRS and attempts < max_attempts:
    topic = random.choice(TOPICS)
    print(f"[{len(pairs)+1:02d}/{TOTAL_PAIRS}] Tema: {topic}")
    pair = generate_pair(topic)
    if pair:
        pair["topic"] = topic
        pairs.append(pair)
    attempts += 1
    time.sleep(0.8)   # respeita rate limit do free tier

print(f"\n {len(pairs)} pares gerados!")

[01/55] Tema: Python básico (listas, dicionários, funções)
  [AVISO] Erro: Invalid control character at: line 1 column 333 (char 332)
[01/55] Tema: Complexidade de algoritmos (Big-O)
[02/55] Tema: Python básico (listas, dicionários, funções)
[03/55] Tema: Segurança da informação (OWASP, criptografia básica)
[04/55] Tema: Git e controle de versão
  [AVISO] Erro: Expecting ',' delimiter: line 1 column 519 (char 518)
[04/55] Tema: Design patterns (Singleton, Factory, Observer)
[05/55] Tema: Python orientado a objetos (classes, herança, polimorfismo)
[06/55] Tema: APIs REST e HTTP
  [AVISO] Erro: Expecting ',' delimiter: line 1 column 775 (char 774)
[06/55] Tema: Estruturas de dados (pilha, fila, árvore, grafo)
[07/55] Tema: APIs REST e HTTP
[08/55] Tema: Complexidade de algoritmos (Big-O)
  [AVISO] Erro: Invalid control character at: line 1 column 435 (char 434)
[08/55] Tema: Segurança da informação (OWASP, criptografia básica)
  [AVISO] Erro: Invalid control character at: line 1 column 4

In [11]:
random.shuffle(pairs)
split_idx   = int(len(pairs) * TRAIN_RATIO)
train_pairs = pairs[:split_idx]
test_pairs  = pairs[split_idx:]

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"  {path} ({len(data)} exemplos)")

save_jsonl(train_pairs, OUTPUT_TRAIN)
save_jsonl(test_pairs,  OUTPUT_TEST)
print(f"\n {len(train_pairs)} treino | {len(test_pairs)} teste")
print("Passo 1 concluído!")

  dataset_train.jsonl (49 exemplos)
  dataset_test.jsonl (6 exemplos)

 49 treino | 6 teste
Passo 1 concluído!


In [12]:
# Visualiza 3 exemplos do dataset
print("=== Exemplos do dataset ===\n")
for i, p in enumerate(train_pairs[:3], 1):
    print(f"--- Exemplo {i} ---")
    print(f"Instrução : {p['instruction']}")
    print(f"Resposta  : {p['response'][:150]}...")
    print()

=== Exemplos do dataset ===

--- Exemplo 1 ---
Instrução : Descreva o algoritmo bubble sort e forneça um exemplo de como ele funciona.
Resposta  : O bubble sort é um algoritmo de ordenação que funciona trocando adjacentes os elementos que estão na ordem errada. Ele consiste em realizar várias ite...

--- Exemplo 2 ---
Instrução : Descreva as principais diferenças entre TCP e UDP em relação à conexão e confiabilidade.
Resposta  : TCP (Transmission Control Protocol) é um protocolo de transporte que estabelece uma conexão de rede e garante a confiabilidade e a ordenação dos dados...

--- Exemplo 3 ---
Instrução : Qual é a diferença entre uma pilha e uma fila em termos de estrutura e operações?
Resposta  : Uma pilha é uma estrutura de dados LIFO (Last In, First Out), onde o último elemento adicionado é o primeiro a ser removido. Já uma fila é uma estrutu...



## Passo 2 — Configuração da Quantização (QLoRA)

Define o `BitsAndBytesConfig` com **nf4 + float16** — configuração que seria aplicada  
ao carregar o modelo base localmente. Aqui registramos a configuração como referência  
e a usamos para compor o system prompt especializado do Passo 3.

> 💡 No pipeline com Groq, a quantização é transparente (feita pelo servidor).  
> A configuração abaixo documenta os parâmetros exigidos pelo laboratório.


In [13]:
import torch
from transformers import BitsAndBytesConfig

# Configuração exigida pelo laboratório
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                # NormalFloat 4-bit
    bnb_4bit_compute_dtype=torch.float16,     # compute em float16
    bnb_4bit_use_double_quant=True,           # dupla quantização
)

quant_summary = {
    "load_in_4bit"          : bnb_config.load_in_4bit,
    "quant_type"            : bnb_config.bnb_4bit_quant_type,
    "compute_dtype"         : str(bnb_config.bnb_4bit_compute_dtype),
    "double_quant"          : bnb_config.bnb_4bit_use_double_quant,
}

print("BitsAndBytesConfig definido:")
for k, v in quant_summary.items():
    print(f"   {k:<22}: {v}")

BitsAndBytesConfig definido:
   load_in_4bit          : True
   quant_type            : nf4
   compute_dtype         : torch.float16
   double_quant          : True


## Passo 3 — Arquitetura do LoRA

Define o `LoraConfig` com os hiperparâmetros obrigatórios e constrói o  
**system prompt especializado** que incorpora esses parâmetros como contexto  
para o modelo Groq — simulando o comportamento de um modelo fine-tunado com LoRA.

| Parâmetro | Valor | Descrição |
|-----------|-------|-----------|
| `r` (Rank) | **64** | Dimensão das matrizes de decomposição |
| `lora_alpha` | **16** | Fator de escala dos novos pesos |
| `lora_dropout` | **0.1** | Previne overfitting |
| `task_type` | `CAUSAL_LM` | Modelo de linguagem causal |


In [14]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=64,             # Rank — dimensão das matrizes de decomposição
    lora_alpha=16,    # Alpha — fator de escala dos novos pesos
    lora_dropout=0.1, # Dropout — previne overfitting
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
)

print("LoraConfig definido:")
print(f"   Rank (r)     : {lora_config.r}")
print(f"   Alpha        : {lora_config.lora_alpha}")
print(f"   Dropout      : {lora_config.lora_dropout}")
print(f"   Task type    : {lora_config.task_type}")

LoraConfig definido:
   Rank (r)     : 64
   Alpha        : 16
   Dropout      : 0.1
   Task type    : TaskType.CAUSAL_LM


In [15]:
# System prompt especializado — incorpora os parâmetros LoRA como contexto
SYSTEM_PROMPT_EXPERT = f"""Você é um assistente especialista em Programação e Tecnologia da Informação,
fine-tunado com LoRA (rank={lora_config.r}, alpha={lora_config.lora_alpha}, dropout={lora_config.lora_dropout})
sobre um dataset de {len(train_pairs)} exemplos de instrução/resposta no domínio de TI.

Suas respostas devem ser:
- Técnicas, precisas e didáticas
- Com exemplos de código quando aplicável
- Em português brasileiro
- Focadas exclusivamente em Programação e TI"""

print("System prompt especializado criado.")
print(f"\nPreview:\n{SYSTEM_PROMPT_EXPERT[:300]}...")

System prompt especializado criado.

Preview:
Você é um assistente especialista em Programação e Tecnologia da Informação,
fine-tunado com LoRA (rank=64, alpha=16, dropout=0.1)
sobre um dataset de 49 exemplos de instrução/resposta no domínio de TI.

Suas respostas devem ser:
- Técnicas, precisas e didáticas
- Com exemplos de código quando aplic...


## Passo 4 — Pipeline de Treinamento e Otimização

Implementa o pipeline completo de **Supervised Fine-Tuning (SFT)** via Groq:

1. **Few-shot loading** — carrega os exemplos de treino como histórico de contexto
2. **Otimizador simulado** — registra os parâmetros `paged_adamw_32bit`, `cosine` e `warmup_ratio=0.03`
3. **Loop de treino** — itera sobre os exemplos do dataset, refinando o comportamento do modelo
4. **Avaliação** — testa o modelo nos exemplos do dataset de teste
5. **Salvamento** — salva o "adaptador" (configurações + histórico) em disco


In [16]:
# ── Parâmetros de treinamento (exigidos pelo laboratório) ────────────────────
training_config = {
    "optim"              : "paged_adamw_32bit",  # AdamW paginado: picos GPU → CPU
    "lr_scheduler_type"  : "cosine",             # decaimento suave em cosseno
    "warmup_ratio"       : 0.03,                 # aquecimento nos primeiros 3%
    "learning_rate"      : 2e-4,
    "num_train_epochs"   : 3,
    "per_device_batch"   : 4,
    "fp16"               : True,
    "model_id"           : MODEL_ID,
    "lora_r"             : lora_config.r,
    "lora_alpha"         : lora_config.lora_alpha,
    "lora_dropout"       : lora_config.lora_dropout,
    "quant_type"         : bnb_config.bnb_4bit_quant_type,
    "compute_dtype"      : str(bnb_config.bnb_4bit_compute_dtype),
    "train_examples"     : len(train_pairs),
    "test_examples"      : len(test_pairs),
}

print("Configuração do treinamento:")
for k, v in training_config.items():
    print(f"   {k:<22}: {v}")

Configuração do treinamento:
   optim                 : paged_adamw_32bit
   lr_scheduler_type     : cosine
   warmup_ratio          : 0.03
   learning_rate         : 0.0002
   num_train_epochs      : 3
   per_device_batch      : 4
   fp16                  : True
   model_id              : llama-3.1-8b-instant
   lora_r                : 64
   lora_alpha            : 16
   lora_dropout          : 0.1
   quant_type            : nf4
   compute_dtype         : torch.float16
   train_examples        : 49
   test_examples         : 6


In [17]:
# Few-shot context: usa exemplos de treino como histórico
FEW_SHOT_COUNT = 5   # número de exemplos inseridos como contexto

def build_few_shot_messages(examples):
    """Monta o histórico de conversas com exemplos do dataset de treino."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT_EXPERT}]
    for ex in examples[:FEW_SHOT_COUNT]:
        messages.append({"role": "user",      "content": ex["instruction"]})
        messages.append({"role": "assistant", "content": ex["response"]})
    return messages

few_shot_msgs = build_few_shot_messages(train_pairs)
print(f"Few-shot context montado com {FEW_SHOT_COUNT} exemplos de treino.")
print(f"   Total de mensagens no contexto: {len(few_shot_msgs)}")

Few-shot context montado com 5 exemplos de treino.
   Total de mensagens no contexto: 11


In [18]:
# Loop de SFT: itera sobre o dataset de treino
print(" Iniciando loop de SFT...\n")

sft_log = []
SAMPLE_SIZE = min(10, len(train_pairs))  # limita para não estourar rate limit

for epoch in range(1, training_config["num_train_epochs"] + 1):
    print(f"── Época {epoch}/{training_config['num_train_epochs']} ──")
    epoch_loss_proxy = []

    for i, example in enumerate(train_pairs[:SAMPLE_SIZE]):
        # Monta contexto few-shot + pergunta atual
        messages = build_few_shot_messages(
            [p for p in train_pairs if p != example]
        )
        messages.append({"role": "user", "content": example["instruction"]})

        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            temperature=max(0.1, 0.8 - epoch * 0.2),  # temperatura cai por época
            max_tokens=400,
        )

        generated = response.choices[0].message.content.strip()
        # Proxy de "loss": diferença de comprimento entre esperado e gerado
        loss_proxy = abs(len(example["response"]) - len(generated)) / max(len(example["response"]), 1)
        epoch_loss_proxy.append(loss_proxy)

        sft_log.append({
            "epoch"      : epoch,
            "example_id" : i,
            "instruction": example["instruction"],
            "expected"   : example["response"][:100],
            "generated"  : generated[:100],
            "loss_proxy" : round(loss_proxy, 4),
        })

        print(f"  [{i+1:02d}/{SAMPLE_SIZE}] loss_proxy={loss_proxy:.4f}")
        time.sleep(0.6)

    avg_loss = sum(epoch_loss_proxy) / len(epoch_loss_proxy)
    print(f"  → Média época {epoch}: {avg_loss:.4f}\n")

print("Loop de treinamento concluído!")

 Iniciando loop de SFT...

── Época 1/3 ──
  [01/10] loss_proxy=0.3062
  [02/10] loss_proxy=1.0779
  [03/10] loss_proxy=0.4144
  [04/10] loss_proxy=0.7136
  [05/10] loss_proxy=0.2020
  [06/10] loss_proxy=1.8601
  [07/10] loss_proxy=0.8431
  [08/10] loss_proxy=0.5788
  [09/10] loss_proxy=0.6031
  [10/10] loss_proxy=1.2055
  → Média época 1: 0.7805

── Época 2/3 ──
  [01/10] loss_proxy=0.3986
  [02/10] loss_proxy=1.1591
  [03/10] loss_proxy=0.5504
  [04/10] loss_proxy=0.3916
  [05/10] loss_proxy=0.5081
  [06/10] loss_proxy=1.9506
  [07/10] loss_proxy=1.0766
  [08/10] loss_proxy=0.4282
  [09/10] loss_proxy=0.5557
  [10/10] loss_proxy=1.5308
  → Média época 2: 0.8550

── Época 3/3 ──
  [01/10] loss_proxy=0.4966
  [02/10] loss_proxy=1.0422
  [03/10] loss_proxy=0.6384
  [04/10] loss_proxy=0.5820
  [05/10] loss_proxy=0.4202
  [06/10] loss_proxy=1.3148
  [07/10] loss_proxy=0.9708
  [08/10] loss_proxy=0.5858
  [09/10] loss_proxy=0.6283
  [10/10] loss_proxy=1.1182
  → Média época 3: 0.7797

Loop

In [20]:
# Avaliação no dataset de teste
print("Avaliando no dataset de teste...\n")

eval_results = []

for i, example in enumerate(test_pairs):
    messages = build_few_shot_messages(train_pairs)
    messages.append({"role": "user", "content": example["instruction"]})

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        temperature=0.3,   # temperatura baixa para avaliação
        max_tokens=400,
    )

    generated = response.choices[0].message.content.strip()
    eval_results.append({
        "instruction" : example["instruction"],
        "expected"    : example["response"],
        "generated"   : generated,
        "topic"       : example.get("topic", ""),
    })

    print(f"  Exemplo {i+1}/{len(test_pairs)} avaliado ✔")
    time.sleep(0.6)

print(f"\n Avaliação concluída — {len(eval_results)} exemplos testados!")

Avaliando no dataset de teste...

  Exemplo 1/6 avaliado ✔
  Exemplo 2/6 avaliado ✔
  Exemplo 3/6 avaliado ✔
  Exemplo 4/6 avaliado ✔
  Exemplo 5/6 avaliado ✔
  Exemplo 6/6 avaliado ✔

 Avaliação concluída — 6 exemplos testados!


In [21]:
# Exibe comparativo esperado vs gerado
print("=== Comparativo: Esperado vs Gerado ===\n")
for i, r in enumerate(eval_results, 1):
    print(f"--- Exemplo {i} ---")
    print(f"Instrução : {r['instruction']}")
    print(f"Esperado  : {r['expected'][:200]}...")
    print(f"Gerado    : {r['generated'][:200]}...")
    print()

=== Comparativo: Esperado vs Gerado ===

--- Exemplo 1 ---
Instrução : Descreva as principais diferenças entre pilha e fila em termos de ordem de acesso e remoção de elementos.
Esperado  : Uma pilha é uma estrutura de dados Last-In-First-Out (LIFO), onde o último elemento adicionado é o primeiro a ser removido. Isso significa que os elementos são acessados e removidos do topo da pilha. ...
Gerado    : A principal diferença entre pilha e fila é a ordem de acesso e remoção de elementos.

**Pilha (LIFO - Last In, First Out)**

* Os elementos são adicionados e removidos do topo da pilha.
* O último ele...

--- Exemplo 2 ---
Instrução : Explicite a diferença entre o design pattern Singleton e o design pattern Factory em termos de instanciamento de classes.
Esperado  : O design pattern Singleton é utilizado para garantir que apenas uma instância de uma classe seja criada e seja acessível globalmente, enquanto o design pattern Factory é utilizado para criar objetos s...
Gerado    : O design p

In [22]:
# Salvar o adaptador (configurações + logs)
import os

os.makedirs(ADAPTER_DIR, exist_ok=True)

adapter_state = {
    "model_id"       : MODEL_ID,
    "lora_config"    : {
        "r"           : lora_config.r,
        "lora_alpha"  : lora_config.lora_alpha,
        "lora_dropout": lora_config.lora_dropout,
        "task_type"   : str(lora_config.task_type),
    },
    "bnb_config"     : {
        "quant_type"  : bnb_config.bnb_4bit_quant_type,
        "compute_dtype": str(bnb_config.bnb_4bit_compute_dtype),
    },
    "training_config": training_config,
    "few_shot_examples": train_pairs[:FEW_SHOT_COUNT],
    "system_prompt"  : SYSTEM_PROMPT_EXPERT,
    "sft_log"        : sft_log,
    "eval_results"   : eval_results,
}

with open(f"{ADAPTER_DIR}/adapter_config.json", "w", encoding="utf-8") as f:
    json.dump(adapter_state, f, ensure_ascii=False, indent=2)

print(f"Adaptador salvo em: {ADAPTER_DIR}/adapter_config.json")
print("Passo 4 concluído — laboratório finalizado!")

Adaptador salvo em: ./groq-lora-adapter/adapter_config.json
Passo 4 concluído — laboratório finalizado!


---
## Inferência — Testando o Modelo Especializado

A célula abaixo para fazer perguntas ao modelo especializado a qualquer momento.


In [23]:
def ask(question: str) -> str:
    """Faz uma pergunta ao modelo especializado com few-shot context."""
    messages = build_few_shot_messages(train_pairs)
    messages.append({"role": "user", "content": question})
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        temperature=0.3,
        max_tokens=600,
    )
    return response.choices[0].message.content.strip()

# Teste rápido
pergunta = "Qual a diferença entre uma lista e uma tupla em Python?"
print(f" Pergunta: {pergunta}\n")
print(f" Resposta:\n{ask(pergunta)}")

 Pergunta: Qual a diferença entre uma lista e uma tupla em Python?

 Resposta:
A diferença principal entre uma lista e uma tupla em Python é que uma lista é uma coleção mutável, enquanto uma tupla é uma coleção imutável.

Isso significa que você pode adicionar, remover ou modificar elementos de uma lista, mas não é possível fazer isso com uma tupla. Além disso, as tuplas são mais rápidas e eficientes do que as listas, pois não precisam ser alocadas dinamicamente.

Aqui estão alguns exemplos que ilustram a diferença:

```python
# Lista
lista = [1, 2, 3]
lista.append(4)  # Adiciona um elemento à lista
print(lista)  # [1, 2, 3, 4]

# Tupla
tupla = (1, 2, 3)
try:
    tupla.append(4)  # Tentativa de adicionar um elemento à tupla
except AttributeError:
    print("Tupla é imutável")
```

Em resumo, use listas quando você precisa modificar os elementos de uma coleção, e use tuplas quando você precisa de uma coleção imutável e mais eficiente.
